# Replicant — Convergence Benchmark Analysis

Loads `results.csv` from the repo root and produces thesis-ready figures in `figures/`.

**Generate results first:**
```sh
cargo run --bin orchestrator -- --trials 10 --output csv \
  scenarios/full-mesh-n{2,3,5,10}.toml \
  scenarios/partition-heal-n{4,6,8}.toml \
  2>/dev/null > results.csv
```

> **If regenerating after a code change:** the CSV is cached as `results.parquet` for
> faster reruns. The cache is refreshed automatically when `results.csv` is newer than
> `results.parquet`. To force a rebuild, delete `results.parquet` manually.

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 150

REPO = Path("..")  # notebook lives in analysis/
CSV     = REPO / "results.csv"
PARQUET = REPO / "results.parquet"
FIGS    = Path("figures")
FIGS.mkdir(exist_ok=True)

## Load data

On first run the CSV is parsed and cached as Parquet (preserves dtypes, loads faster on reruns).

In [ ]:
csv_mtime = CSV.stat().st_mtime if CSV.exists() else 0
parquet_fresh = PARQUET.exists() and PARQUET.stat().st_mtime >= csv_mtime

if parquet_fresh:
    df = pd.read_parquet(PARQUET)
else:
    df = pd.read_csv(CSV)
    df.to_parquet(PARQUET)

trials  = df[df.row_type == "trial"].copy()
summary = df[df.row_type == "summary"].copy()
# In summary rows the `trial` column holds the trial count, not a trial number.
summary = summary.rename(columns={"trial": "n_trials"})

print(f"{len(trials)} trial records, {trials.scenario.nunique()} scenarios, "
      f"{trials.groupby('scenario').size().iloc[0]} trials each")
summary[["scenario", "n_trials", "node_count", "op_count", "mean_ms", "p50_ms", "p95_ms"]]

## Full-mesh: convergence vs node count

Measures time from last write to all-nodes fingerprint agreement.
Shaded band shows p50–p95 range across trials.

In [ ]:
mesh = summary[summary.scenario.str.startswith("full-mesh")].sort_values("node_count")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(mesh.node_count, mesh.mean_ms, "o-", label="mean")
ax.fill_between(mesh.node_count, mesh.p50_ms, mesh.p95_ms, alpha=0.2, label="p50–p95")
ax.set_xlabel("Node count")
ax.set_ylabel("Convergence (ms)")
ax.set_title("Full-mesh: convergence latency vs N")
ax.legend()
fig.tight_layout()
fig.savefig(FIGS / "full_mesh_scaling.pdf")
plt.show()

## Partition-heal: convergence vs partition depth

Measures time from heal trigger (cross-group edges added) to global convergence.
`ConnectPeer` blocks until the Automerge sync handshake is open (via an internal
readiness signal), so these numbers reflect actual CRDT merge cost rather than
any fixed settle delay.

In [ ]:
heal = summary[summary.scenario.str.startswith("partition")].sort_values("node_count")

fig, ax = plt.subplots(figsize=(6, 4))
x = range(len(heal))
ax.bar(x, heal.mean_ms, yerr=heal.p95_ms - heal.mean_ms, capsize=5, alpha=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(heal.scenario, rotation=15, ha="right")
ax.set_ylabel("Heal convergence (ms)")
ax.set_title("Partition-heal: convergence by partition depth")
fig.tight_layout()
fig.savefig(FIGS / "partition_heal.pdf")
plt.show()

## Raw distributions (box plots)

Shows the full trial distribution rather than summary statistics.
More honest for small N — outliers are visible.

In [ ]:
order = (
    trials.groupby("scenario")["convergence_ms"]
    .median()
    .sort_values()
    .index
)

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=trials, x="scenario", y="convergence_ms", order=order, ax=ax)
ax.tick_params(axis="x", rotation=25)
ax.set_xlabel(None)
ax.set_ylabel("Convergence (ms)")
ax.set_title("Convergence distribution per scenario")
fig.tight_layout()
fig.savefig(FIGS / "boxplot.pdf")
plt.show()

## Summary table

Formatted for copy-paste into the thesis evaluation section.

In [ ]:
table = summary[["scenario", "node_count", "n_trials", "mean_ms", "p50_ms", "p95_ms"]].copy()
table.columns = ["Scenario", "Nodes", "Trials", "Mean (ms)", "p50 (ms)", "p95 (ms)"]
table = table.set_index("Scenario")
table.round(1)

## Measurement stability

Standard deviation and coefficient of variation (CV = σ/μ) across trials.
CV < 10% is generally stable; CV > 30% suggests the measurement is noisy
and more trials or a quieter environment are needed.

In [ ]:
stability = (
    trials.groupby("scenario")["convergence_ms"]
    .agg(mean="mean", std="std", n="count")
    .assign(cv_pct=lambda df: (df["std"] / df["mean"] * 100).round(1))
    .round({"mean": 2, "std": 2})
    .sort_values("cv_pct", ascending=False)
)
stability.columns = ["Mean (ms)", "Std (ms)", "N trials", "CV (%)"]

# Highlight rows where CV exceeds 20% — worth investigating
def highlight_cv(row):
    return ["background-color: #fdd" if row["CV (%)"] > 20 else "" for _ in row]

stability.style.apply(highlight_cv, axis=1)

## OTel Protocol Metrics

Loaded from `metrics.json` written by the orchestrator when `--metrics-file` is passed.

OTel counters accumulate across every scenario in a single run, so **run one scenario
at a time** to get per-scenario metrics:

```sh
for s in full-mesh-n2 full-mesh-n3 full-mesh-n5 full-mesh-n10 \
          partition-heal-n4 partition-heal-n6 partition-heal-n8; do
  cargo run --bin orchestrator -- --trials 10 --output csv \
    --metrics-file "metrics-${s}.json" \
    "scenarios/${s}.toml" \
    2>/dev/null >> results.csv
done
```

Then point `METRICS` below at whichever file you want to inspect, e.g.
`REPO / "metrics-full-mesh-n5.json"`.

> **Multi-scenario (cumulative) alternative** — useful only as a sanity-check
> that the overall protocol traffic looks reasonable; values cannot be attributed
> to individual scenarios:
> ```sh
> cargo run --bin orchestrator -- --trials 10 --output csv \
>   --metrics-file metrics-all.json \
>   scenarios/full-mesh-n{2,3,5,10}.toml \
>   scenarios/partition-heal-n{4,6,8}.toml \
>   2>/dev/null > results.csv
> ```

In [ ]:
import json

# Point this at a single-scenario metrics file for per-scenario analysis,
# e.g. REPO / "metrics-full-mesh-n5.json".
# Use REPO / "metrics-all.json" for the cumulative multi-scenario view.
METRICS = REPO / "metrics.json"


def load_metrics(path: Path) -> dict[str, pd.DataFrame] | None:
    """Parse a metrics JSON Lines file into a dict of DataFrames keyed by metric name.

    Each line in the file is a JSON object ``{"metrics": [...]}``.
    Data points from multiple flushes are concatenated per metric name.
    Returns None if the file does not exist.
    """
    if not path.exists():
        print(f"[metrics] {path} not found — run the per-scenario loop above first.")
        return None

    rows: list[dict] = []
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    by_name: dict[str, list[dict]] = {}
    for record in rows:
        for m in record.get("metrics", []):
            by_name.setdefault(m["name"], []).extend(m["data_points"])

    return {name: pd.DataFrame(points) for name, points in by_name.items()}


metrics = load_metrics(METRICS)
if metrics:
    print("Loaded metrics:", list(metrics.keys()))
    for name, df_m in metrics.items():
        print(f"  {name}: {len(df_m)} data points, columns={list(df_m.columns)}")

### Sync message traffic

Total Automerge sync messages sent and received per node.
Shows how much protocol chatter each node generates — useful for arguing O(N²) scaling of the gossip layer.


In [ ]:
if metrics:
    tx = metrics["replicant.sync.messages.tx"].groupby("actor")["value"].sum().rename("tx")
    rx = metrics["replicant.sync.messages.rx"].groupby("actor")["value"].sum().rename("rx")
    traffic = pd.concat([tx, rx], axis=1).fillna(0).astype(int)
    # Sort nodes naturally (node-0, node-1, …)
    traffic = traffic.loc[sorted(traffic.index, key=lambda s: int(s.split("-")[1]))]

    fig, ax = plt.subplots(figsize=(max(4, len(traffic) * 0.9), 4))
    x = range(len(traffic))
    width = 0.35
    ax.bar([i - width / 2 for i in x], traffic["tx"], width, label="sent (tx)", alpha=0.8)
    ax.bar([i + width / 2 for i in x], traffic["rx"], width, label="received (rx)", alpha=0.8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(traffic.index)
    ax.set_xlabel("Node")
    ax.set_ylabel("Sync messages")
    ax.set_title("Sync message traffic per node (all scenarios, cumulative)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGS / "sync_traffic.pdf")
    plt.show()
    print(traffic.assign(total=traffic.tx + traffic.rx).sort_values("total", ascending=False))


### Op application latency

Per-node mean latency for `map_put` operations (from the histogram `sum / count`).
Measures the cost of Automerge's local commit — independent of network.


In [ ]:
if metrics:
    op_df = metrics["replicant.op.duration"].copy()
    op_df["mean_ms"] = op_df["sum"] / op_df["count"]
    op_df = op_df.sort_values("actor", key=lambda s: s.map(lambda v: int(v.split("-")[1])))

    fig, ax = plt.subplots(figsize=(max(4, len(op_df) * 0.9), 4))
    bars = ax.bar(op_df["actor"], op_df["mean_ms"], alpha=0.8)
    if "min" in op_df.columns and "max" in op_df.columns:
        ax.errorbar(
            op_df["actor"],
            op_df["mean_ms"],
            yerr=[op_df["mean_ms"] - op_df["min"], op_df["max"] - op_df["mean_ms"]],
            fmt="none",
            color="black",
            capsize=4,
        )
    ax.set_xlabel("Node")
    ax.set_ylabel("Mean op latency (ms)")
    ax.set_title("Automerge map_put latency per node (min/mean/max)")
    fig.tight_layout()
    fig.savefig(FIGS / "op_latency.pdf")
    plt.show()
    print(op_df[["actor", "count", "min", "mean_ms", "max"]].to_string(index=False))


### Document size after convergence

Serialized Automerge document size (bytes) per node after all ops are applied.
All nodes should hold the same logical state — size differences reveal any causal history divergence.


In [ ]:
if metrics:
    doc_df = metrics["replicant.doc.size_bytes"].copy()
    doc_df = doc_df.sort_values("actor", key=lambda s: s.map(lambda v: int(v.split("-")[1])))

    fig, ax = plt.subplots(figsize=(max(4, len(doc_df) * 0.9), 4))
    ax.bar(doc_df["actor"], doc_df["value"], alpha=0.8)
    ax.set_xlabel("Node")
    ax.set_ylabel("Document size (bytes)")
    ax.set_title("Serialized Automerge document size per node")
    fig.tight_layout()
    fig.savefig(FIGS / "doc_size.pdf")
    plt.show()
    print(doc_df[["actor", "value"]].rename(columns={"value": "bytes"}).to_string(index=False))
